# 02 · Policies

**What you will see**

- `Policy(...)` is a factory that picks the provider from the **form** of what you pass, never from a family name.
- A plain function drives the SO-101 arm.
- `Policy("holosoma")` keeps its G1 neighbour standing; `Policy("microduck?vx=0.3")` walks a small duck.
- A form it does not know is refused with the list of forms it does know.
- A real (tiny, random) LeRobot ACT checkpoint drives the arm **in-process**, then the **same checkpoint** is served by LeRobot's own policy server and drives the arm **over Zenoh**; the first served action equals the local one.
- How units are named (`semantics.json`), and the GPU policy matrix measured on an L40S.

Every policy plugs into the same loop: observe → act → write.

In [1]:
import math, socket, sys, tempfile, time
from pathlib import Path
import numpy as np
import _nb
from strands_robots import Description, Policy, Robot, Simulation
_nb.show(_nb.drawing("d03_one_loop"))

## The forms

`Policy(spec)` looks at the shape of `spec`. These are all of them:

In [2]:
from strands_robots.policy.factory import FORMS
print("\n".join(FORMS))

<preset>[?k=v&k2=v2]            a named ONNX preset (holosoma, microduck), run in-process
zenoh://host:port/<deployment>   a lerobot-policy-server
<owner>/<repo>                   a LeRobot checkpoint on the Hugging Face Hub, run in-process
<dir with config.json>           a local LeRobot checkpoint, run in-process
<callable>                       f(obs) -> action


## A function is a policy

`Policy(callable)` wraps `f(obs) -> action`. The action can name only the joints it moves. An unattached camera (`view`) is just for us to watch.

In [3]:
arm = Robot("so101")
arm.sim.camera("view", position=[0.45, -0.45, 0.35], target=[0, 0, 0.12], size=(320, 240))
wave = Policy(lambda o: {"shoulder_pan": 0.8 * math.sin(3 * o.t), "elbow_flex": 0.5 * math.sin(2 * o.t)})
frames = []
r = arm.run(wave, hz=30, seconds=3, stop_when=lambda o: frames.append(arm.sim.render("view")) or False)
print(type(wave).__name__, "|", r.steps, "steps,", r.stopped_by, "| final pan", round(float(r.final.joints.positions[0]), 3), "rad")
arm.sim.close()
_nb.video(frames, fps=30, width=480)

CallablePolicy | 90 steps, steps | final pan 0.377 rad


## A named ONNX preset: the G1 stands

`Policy("holosoma")` is a whole-body ONNX controller for the Unitree G1 (weights are hash-checked, run with onnxruntime in-process). It runs at 50 Hz; we watch the pelvis height.

In [4]:
def watch(robot, sim, policy, seconds, every=5):
    frames, zs, x0 = [], [], float(robot.pose.position[0])
    def tick(o):
        zs.append(float(o.pose.position[2]))
        if len(zs) % every == 1:
            frames.append(sim.render("cam"))
        return False
    res = robot.run(policy, seconds=seconds, stop_when=tick)
    return res, np.array(zs), float(robot.pose.position[0]) - x0, frames

sim = Simulation("mujoco", render_size=(400, 300))
g1 = Robot("g1", sim=sim, keyframe="stand")
sim.camera("cam", position=(2.0, -2.0, 1.4), target=(0.0, 0.0, 0.75), size=(400, 300))
res, zs, _, frames = watch(g1, sim, Policy("holosoma"), seconds=8)
print(f"{res.steps} steps at {res.hz_target:g} Hz, {res.stopped_by} | pelvis min {zs.min():.3f} m "
      f"(t={zs.argmin() / res.hz_target:.2f} s, the start-up knee bend), final {zs[-1]:.3f} m")
sim.close()
_nb.video(frames, fps=10, width=480)

400 steps at 50 Hz, steps | pelvis min 0.742 m (t=0.32 s, the start-up knee bend), final 0.765 m


## Parameters ride in the spec: the duck walks

`"microduck?vx=0.3"` is the same as `Policy("microduck", vx=0.3)`: a forward velocity command of 0.3 m/s.

In [5]:
sim = Simulation("mujoco", render_size=(400, 300))
duck = Robot("microduck", sim=sim)
sim.camera("cam", position=(0.65, -1.9, 0.7), target=(0.65, 0.0, 0.05), size=(400, 300))
res, zs, dx, frames = watch(duck, sim, Policy("microduck?vx=0.3"), seconds=8)
print(f"{res.steps} steps, {res.stopped_by} | walked dx {dx:.2f} m in 8 s | height min {zs.min():.3f} m")
sim.close()
_nb.video(frames, fps=10, width=480)

400 steps, steps | walked dx 1.03 m in 8 s | height min 0.107 m


## An unknown form is refused, naming what is known

Refused before anything is downloaded or loaded.

In [6]:
for spec, kw in [("nope", {}), ("microduck?vz=1", {}), ("microduck", {"bogus": 1}), (42, {})]:
    try:
        Policy(spec, **kw)
    except (ValueError, TypeError) as e:
        print(f"Policy({spec!r}{', **' + str(kw) if kw else ''}) -> {type(e).__name__}: {str(e).splitlines()[0]}")

Policy('nope') -> ValueError: Policy('nope'): unknown form; known presets: holosoma, microduck; forms:
Policy('microduck?vz=1') -> ValueError: microduck: unknown parameter 'vz'; known: skill, vx, vy, wz
Policy('microduck', **{'bogus': 1}) -> TypeError: microduck: unexpected keyword 'bogus'; known: skill, vx, vy, wz
Policy(42) -> TypeError: Policy(int) is not a policy spec; forms:


## LeRobot, in-process

A directory with a `config.json` is a LeRobot checkpoint. We build a real ACT checkpoint with random weights (no download, about 1 MB) that expects the six SO-101 joints and one 64×48 camera called `front`. LeRobot loads it with its saved processors; strands only maps names, units and cameras at the edge.

In [7]:
sys.path.insert(0, str(_nb.ROOT / "tests" / "policy"))
from tiny_act import save_tiny_act
work = Path(tempfile.mkdtemp(prefix="nb02-"))
ckpt = str(save_tiny_act(work / "tiny-act-so101", 6, camera="front"))
print(sorted(p.name for p in Path(ckpt).iterdir()))

def so101():
    a = Robot("so101")
    a.sim.camera("front", size=(64, 48), attach=a)                          # what the policy sees
    a.sim.camera("view", position=[0.45, -0.45, 0.35], target=[0, 0, 0.12], size=(320, 240))  # what we watch
    return a

arm = so101()
local_policy = Policy(ckpt, device="cpu")
frames = []
local = arm.run(local_policy, seconds=2, stop_when=lambda o: frames.append(arm.sim.render("view")) or False)
print(type(local_policy).__name__, "around", type(local_policy.inner).__name__)
print("local RunResult:", f"steps={local.steps} hz_target={local.hz_target} stopped_by={local.stopped_by!r} fault={local.fault}")
arm.sim.close()
_nb.video(frames, fps=30, width=480)

['config.json', 'model.safetensors', 'policy_postprocessor.json', 'policy_postprocessor_step_0_unnormalizer_processor.safetensors', 'policy_preprocessor.json', 'policy_preprocessor_step_3_normalizer_processor.safetensors']


Loading weights from local directory


ChunkedPolicy around LeRobotPolicy
local RunResult: steps=60 hz_target=30.0 stopped_by='steps' fault=None


## The server config is built, not hand-written

`server_config(description, checkpoint)` reads the checkpoint's features and the robot's registry entry: state/action names are `<joint>.pos`, cameras come at their trained size, `robot_type` and the units label (`semantics`) come from the registry. `write_server_yaml` writes the exact YAML `lerobot-policy-server --config_path=` serves.

In [8]:
from strands_robots.policy.lerobot import serve, server_config, write_server_yaml

def free_endpoint(ports=range(8745, 8750)):
    for port in ports:                      # this notebook's port range
        with socket.socket() as s:
            try:
                s.bind(("127.0.0.1", port))
            except OSError:
                continue
        return f"tcp/127.0.0.1:{port}"
    raise RuntimeError("ports 8745-8749 are all busy")

cfg = server_config(Description.load("so101"), ckpt, deployment="tiny", endpoint=free_endpoint(), device="cpu")
print("robot_type:", cfg.robot_type, "| semantics:", cfg.semantics)
print("action names:", cfg.action_feature.names)
y = write_server_yaml(cfg, work / "tiny.yaml").read_text().splitlines()
print("--- tiny.yaml (excerpt) ---")
y = [l.replace(str(work), "<work>") for l in y]
cut = next(i for i, l in enumerate(y) if l.startswith("semantics:"))
print("\n".join(y[:5] + ["..."] + y[cut:cut + 30] + ["..."]))

robot_type: so101_follower | semantics: so101-range-gripper-percent-v1
action names: ('shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos', 'wrist_flex.pos', 'wrist_roll.pos', 'gripper.pos')
--- tiny.yaml (excerpt) ---
deployment: tiny
model:
  repo_or_path: <work>/tiny-act-so101
  revision: null
  device: cpu
...
semantics: so101-range-gripper-percent-v1
robot_type: so101_follower
features:
- name: observation.state
  shape:
  - 6
  dtype: float32
  kind: tensor
  names:
  - shoulder_pan.pos
  - shoulder_lift.pos
  - elbow_flex.pos
  - wrist_flex.pos
  - wrist_roll.pos
  - gripper.pos
  semantics: so101-range-gripper-percent-v1
- name: observation.images.front
  shape:
  - 48
  - 64
  - 3
  dtype: uint8
  kind: rgb
  names: []
  semantics: rgb-v1
action_feature:
  name: action
  shape:
  - 6
  dtype: float32
...


## The same checkpoint, served over Zenoh

`serve(cfg)` runs LeRobot's own `PolicyServer` in this process (the same code path as `lerobot-policy-server`) and yields a `zenoh://host:port/<deployment>` URL. `Policy(url)` is a LeRobot remote client. The run is paced in wall time (LeRobot's clock). We record what was sent and what came back, then check parity: the first served action (past the start-up hold) equals what the local checkpoint computes on a frame that was sent. The server is stopped when the `with` block ends.

In [9]:
from strands_robots.policy.lerobot import LeRobotPolicy, ZenohPolicy
from lerobot.inference import ObservationSnapshot

sent, got = [], []
real_act = ZenohPolicy.act
def spy(self, obs, *, instruction=None):
    sent.append(self.contract.frame(obs))
    out = real_act(self, obs, instruction=instruction)
    got.append(np.array(out))
    return out

with serve(cfg) as url:
    print("serving", url)
    remote_policy = Policy(url, chunk_merge="append")   # append: the first served action is chunk[0] of its frame
    print(type(remote_policy).__name__, remote_policy.describe()["semantics"])
    ZenohPolicy.act = spy
    try:
        arm = so101()
        frames = []
        t = time.perf_counter()
        remote = arm.run(remote_policy, seconds=2, stop_when=lambda o: frames.append(arm.sim.render("view")) or False)
        wall = time.perf_counter() - t
    finally:
        ZenohPolicy.act = real_act

print("remote RunResult:", f"steps={remote.steps} hz_target={remote.hz_target} stopped_by={remote.stopped_by!r} fault={remote.fault} wall={wall:.2f}s")

check = LeRobotPolicy(ckpt, device="cpu")
check.attach(arm)
first = next(i for i, a in enumerate(got) if not np.allclose(a, got[0]))   # first action past the start-up hold
diff = min(np.abs(check.contract.to_keys_chunk(check._runner.predict(
           ObservationSnapshot(f, time.monotonic(), "", 0, "parity")).canonical_actions.numpy())[0] - got[first]).max()
           for f in sent[: first + 1])
print(f"parity: first served action (step {first}) vs local checkpoint: max |diff| = {diff:.2e} rad")
assert diff < 1e-4
_nb.table([
    {"run": "in-process", "steps": local.steps, "stopped_by": local.stopped_by, "fault": local.fault},
    {"run": "over Zenoh", "steps": remote.steps, "stopped_by": remote.stopped_by, "fault": remote.fault},
])

Loading weights from local directory
serving zenoh://127.0.0.1:8745/tiny


ZenohPolicy so101-range-gripper-percent-v1


remote RunResult: steps=60 hz_target=30.0 stopped_by='steps' fault=None wall=2.15s
Loading weights from local directory


parity: first served action (step 1) vs local checkpoint: max |diff| = 0.00e+00 rad


| run | steps | stopped_by | fault |
|---|---|---|---|
| in-process | 60 | steps | None |
| over Zenoh | 60 | steps | None |

In [10]:
arm.sim.close()
_nb.video(frames, fps=30, width=480)

## Units: what the numbers mean

A LeRobot deployment names its units with a `semantics` string. LeRobot keeps no table of these; strands does, as data in `units/semantics.json`. The SO-101 default is LeRobot's range mode (−100…100, gripper 0…100); a degrees checkpoint gets the degrees label. Inside strands everything is radians; conversion happens only at the edge.

In [11]:
import json
from strands_robots.units import semantics_for, units_for_semantics
print(json.dumps({k: v for k, v in json.loads((_nb.ROOT / "strands_robots/units/semantics.json").read_text()).items()
                  if k.startswith(("radians", "so101"))}, indent=1))
desc = Description.load("so101")
q = np.array([0.5, -0.3, 0.2, 0.0, 0.1, 0.8])   # radians
rows = []
for norm in (None, {"*": "degrees", "gripper": "range_0_100"}):
    sem = semantics_for(desc, norm)
    u = units_for_semantics(sem, desc)
    rows.append({"norm": norm or "registry default", "semantics": sem, "units": u.name,
                 "q (rad) ->": np.round(u.from_rad(q), 2).tolist(), "round trip ok": bool(np.allclose(u.to_rad(u.from_rad(q)), q))})
_nb.table(rows)

{
 "radians-v1": {
  "units": "rad"
 },
 "so101-degrees-gripper-percent-v1": {
  "lerobot_norm": {
   "*": "degrees",
   "gripper": "range_0_100"
  }
 },
 "so101-range-gripper-percent-v1": {
  "lerobot_norm": {
   "*": "range_m100_100",
   "gripper": "range_0_100"
  }
 }
}


| norm | semantics | units | q (rad) -> | round trip ok |
|---|---|---|---|---|
| registry default | so101-range-gripper-percent-v1 | lerobot | [26.04, -17.19, 17.33, 0.0, 3.58, 50.76] | True |
| {'*': 'degrees', 'gripper': 'range_0_100'} | so101-degrees-gripper-percent-v1 | lerobot | [28.65, -17.19, 16.46, 0.0, 5.73, 50.76] | True |

## Every LeRobot family over Zenoh (measured on an L40S, not re-run here)

Each row: a real `lerobot-policy-server` subprocess with the YAML from `server_config`, driving a strands sim body for 11 s at 30 Hz. Contract level only (admitted, finite, inside joint limits): a row is never a task-success claim. Refusals are LeRobot's or strands' own messages, before any motion.

In [12]:
md = (_nb.MEDIA / "gpu-matrix/MATRIX.md").read_text().splitlines()
start = next(i for i, l in enumerate(md) if l.startswith("| family"))
head = [c.strip() for c in md[start].strip("|").split("|")]
keep = ["family", "checkpoint", "robot", "status", "turnaround_p50_ms", "hz", "steps", "inside_limits", "vram_mib", "parity_max_abs_diff"]
rows = []
for l in md[start + 2:]:
    if not l.startswith("|"):
        break
    r = dict(zip(head, [c.strip() for c in l.strip("|").split("|")]))
    rows.append({k: r.get(k, "") for k in keep})
print(len(rows), "rows,", sum(r["status"] == "admitted" for r in rows), "admitted")
_nb.table(rows)

28 rows, 12 admitted


| family | checkpoint | robot | status | turnaround_p50_ms | hz | steps | inside_limits | vram_mib | parity_max_abs_diff |
|---|---|---|---|---|---|---|---|---|---|
| ACT (core) | alexis779/so101_ball_cup_act@31ff9e7e | so101 | admitted | 18.5 | 30.0 | 330 | True | 816 | 0.0 |
| Diffusion (core) | Lebruhbruh/diffusion_so101_pick_cube@f38d5537 | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| DM05 (dm05) | alexhegit/so101-simstudio-lab01-pnp-dm05-agent193-10k@a02640ce | so101 | fault | 690.1 | 29.9 | 23 | True | 20288 |  |
| DM05 (dm05) | alexhegit/so101-simstudio-lab01-pnp-dm05-agent193-10k@a02640ce | so101 | admitted | 651.5 | 8.0 | 88 | True | 20288 |  |
| EO1 (eo1) | crislmfroes/eo1-openarm-lift-fanta-bottle-clean@74fcfe47 | openarm | refused (strands, server_config) |  |  |  |  |  |  |
| EVO1 (evo1) | quangnd58/evo1-so101-multitask@5b6d832c | so101 | refused (strands, server_config) |  |  |  |  |  |  |
| EVO1 (evo1) | imstevenpmwork/super_chatton_evo1_stage1_pro@80cd23cd | so101 | admitted | 293.7 | 30.0 | 330 | True | 4386 |  |
| FastWAM (fastwam) | H2Ozone/fastwam_new_green_2026-07-19_16-26-08@239e1573 | so101 | refused (LeRobot chunk_inference_spec, config-level) |  |  |  |  |  |  |
| FineART (fineart_vla) | pepijn223/fineart_vla_super_poulain@ddb98404 | so101 | admitted | 238.0 | 30.0 | 330 | True | 18856 |  |
| FLUX.3 | 5hadytru/so101_bench_flux-3-action-so101@ac49157c | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| G1 SONIC (unitree_g1) | nepyope/sonic_walk@c37a5e42 | unitree_g1 | refused (strands, server_config) |  |  |  |  |  |  |
| Gaussian (gaussian_actor) | tiny:gaussian_actor (random, LeRobot defaults)@ | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| GR00T (groot) | fatdove/so101-cube-bowl_GR00T17@71dc7eae | so101 | admitted | 172.4 | 30.0 | 330 | True | 13698 |  |
| LaWAM (lawam) | 5hadytru/so101_bench_LaWAM_full_fine_tune@a7ddd367 | so101 | admitted | 213.0 | 30.0 | 330 | True | 7566 |  |
| LingBot-VA | lerobot/lingbot_va_base@a9b1da7a | aloha | refused (lerobot-policy-server) |  |  |  |  |  |  |
| MolmoAct2 (molmoact2) | XvKuoMing/so101_chess_molmoact2@136a2462 | so101 | admitted | 244.5 | 30.0 | 330 | True | 14942 |  |
| Multi-task DiT | seriintan/multi_task_dit_frazier@7c38d407 | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| π0 (pi) | tonghuiwang123/so100-pi0-new40@54c1c770 | so100 | admitted | 225.2 | 30.0 | 330 | True | 9474 |  |
| π0.5 (pi) | pdhinaka/pi05_so101_blue_cube_gray_box_cleaned_20k@ffbee39f | so101 | admitted | 262.0 | 30.0 | 330 | True | 9858 |  |
| π0-FAST (pi) | gpudad/pi0fast-so101-pick-cube@44d2a385 | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| SmolVLA (smolvla) | lerobot/smolvla_base@5e8d12a6 | so101 | admitted | 241.6 | 30.0 | 330 | True | 1466 |  |
| SmolVLA (smolvla) | lerobot/smolvla_base@5e8d12a6 | so101 | ran |  | 158.4 | 330 | True | 929 |  |
| SmolVLA (smolvla) | lerobot/smolvla_base@5e8d12a6 | so101 | admitted | 360.0 | 30.0 | 330 | True | 2122 |  |
| TD-MPC (core) | tiny:tdmpc (random, LeRobot defaults)@ | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| VLA-JEPA (vla_jepa) | charlott-rgb/so101_acrylic_vla_jepa_b8_15000@1e612363 | so101 | admitted | 105.3 | 30.0 | 330 | True | 6562 |  |
| VQ-BeT (core) | tgibbons/SO-101-VQBET-ball2-claw_30k@5bb580f9 | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| WALL-X (wallx) | imstevenpmwork/wall_x_super_chatton@de41b2df | so101 | refused (lerobot-policy-server) |  |  |  |  |  |  |
| XVLA (xvla) | imstevenpmwork/xvla_super_chatton_2@36fbf138 | so101 | admitted* | 107.6 | 30.0 | 330 | True | 2340 |  |

## A real VLA on your own machine (not executed here)

SmolVLA, served by LeRobot and driven by the arm. It wants three 256×256 cameras named `camera1..3`. Terminal 1:

```bash
python -c "
from strands_robots import Description
from strands_robots.policy.lerobot import server_config, write_server_yaml
write_server_yaml(server_config(Description.load('so101'), 'lerobot/smolvla_base',
                  deployment='smolvla', endpoint='tcp/127.0.0.1:7447'), 'smolvla.yaml')"
lerobot-policy-server --config_path=smolvla.yaml
```

Terminal 2:

```python
from strands_robots import Policy, Robot
arm = Robot("so101")
for i, p in enumerate([(0.6, 0, 0.5), (0, 0.6, 0.5), (0.4, -0.4, 0.6)]):
    arm.sim.camera(f"camera{i + 1}", size=(256, 256), position=p, target=(0, 0, 0.1), attach=arm)
print(arm.run(Policy("zenoh://127.0.0.1:7447/smolvla"), instruction="pick up the cube", seconds=10))
```

On a busy laptop the server's turnaround can exceed the chunk horizon; the run then ends as `stopped_by="fault"` and the arm holds.

## Where this sits in v1

`Policy` is one of the five types (Robot, Simulation, Policy, Description, Observation/Action). Providers live in `strands_robots/policy/`: `callable`, `onnx/` (named presets), `lerobot/` (in-process checkpoints, the Zenoh client, `server_config`, `serve`) with `chunked` handling action chunks. Every one of them is driven by the same `robot.run(...)` loop, on a simulated or a real arm. Next: **03 · Recording** turns those runs into LeRobot datasets.